<a href="https://colab.research.google.com/github/Leooon726/vibecoding/blob/main/%E6%97%B6%E7%A9%BA%E9%87%87%E6%A0%B7%E8%80%85v10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install EbookLib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.3 MB/s eta 0:00:00


In [ ]:
# @title 1. 全局配置与核心提示词库 (User Configuration)

# ==========================================
# A. 基础参数设置 (Basic Settings)
# ==========================================

# 1. API 设置
SILICON_FLOW_API_KEY = "sk-vlmhbxgjgllzolnsqunigerenwtwdfsutvaecdpgpvxqyncc" # @param {type:"string"}
MODEL_NAME = "Qwen/Qwen3-235B-A22B-Instruct-2507" # @param {type:"string"}
MAX_OUTPUT_TOKENS = 8192 # @param {type:"integer"}

# 2. 内容设置
TOPIC = "拉脱维亚从古至今的历史" # @param {type:"string"}
NUM_CHAPTERS = 10 # @param {type:"integer"}
WORDS_PER_CHAPTER = 2000 # @param {type:"integer"}

# 3. 运行设置
DEBUG_MODE = True # @param {type:"boolean"}
MAX_WORKERS = 10 # @param {type:"integer"}

# ==========================================
# B. 提示词库 (Prompt Library) - 核心大脑
# ==========================================

PROMPT_LIBRARY = {
    # 1. 全局系统设定 (System Prompt) - v14.0 保持不变
    "SYSTEM_PROMPT": """
你正在撰写一部**《沉浸式时空历史实录》**。

【1. 核心宗旨：寓教于乐】
* **双重目标**：
  - **宏观线（全球大历史观）**：不仅仅讲述单一对象，更要建立**“全球共时性”**。让读者看到我们所处的角落与世界文明（贸易/思潮/技术）的**紧密连接**。
  - **微观线（物理触感）**：展现真实的社会风貌。**必须包含具体的“饮食起居”与“下三路”细节，赋予大历史以粗糙的真实感。**
* **写作策略**：用精彩的亲历故事承载硬核知识。

【2. 世界观与底层逻辑】
* **缘起**：未来人类患有“空心病”，阿强和K博士肉身穿越采集“情感疫苗”。
* **法则**：
  - **肉身真实性**：阿强在大历史面前是渺小的，这种**“真实在场感”**是故事张力的来源。
  - **技术无摩擦**：严禁在设备故障上浪费笔墨。

【3. 角色定位与动态】
* **阿强（第一人称/执行者）**：
  - **任务驱动**：**【核心逻辑】** 他的行动是为了采集**“高浓度情感样本”**。这需要他深入历史的**高光或至暗时刻**（无论是极致的荣耀还是残酷的生存），只有**高强度的体验**才能完成采样。
  - **高权限视野**：伪装成核心人物身边的“透明人”，见证关键决策。
  - **现代参照系**：关注生活质量与制度观念的直接反差。
* **K博士（坐标建立者/深度旁白）**：
  - **历史观建立**：时刻提醒阿强**“这一刻在人类文明长河中的位置”**。
  - **深度解析**：博士应**高频**介入，作为“画外音”揭示眼前景象背后的**前因后果**（以知促情）。

【4. 叙事基调】
* **大历史的主场**：故事焦点在**关键节点**。微观细节是背景纹理。
""",

    # 2. 史料挖掘指令 (The Historian) - v14.0 保持不变
    "HISTORIAN_PROMPT": """
你是历史架构师。请为《{topic}》构建**【关键历史节点清单】**（{num_chapters} 个）：

1. **宏观锚点**：
   - **历史大事件**：选取该时期决定性的瞬间。
   - **核心人物**：事件的绝对主导者。
   - **全球文明参照**：此时的世界主要文明发生了什么**共振事件**（如：黑死病蔓延、火药传播、启蒙运动）？

2. **硬核知识包**：
   - **专有名词**：3-5个术语（法案、兵种、货币）。
   - **关键数据**：体现真实感的数据。

3. **双重反差素材**：
   - **认知反差**：反现代直觉的社会规则。
   - **生活纹理 (Sensory Details)**：挖掘当时的**具体生存细节**（如饮食口感、居住气味、卫生条件）。**注意：如实还原历史的粗糙感，不做美化。**

【输出格式】：
请直接输出整理好的文本，分点陈述，不要写代码。
""",

    # 3. 大纲规划指令 (The Architect) - v14.0 保持不变
    "ARCHITECT_PROMPT": """
你是总编剧。请设计 {num_chapters} 个**【宏大叙事与任务驱动结合的大纲】**。

【JSON 核心字段要求 (精简版)】：
1. **chapter_id**: 章节序号。
2. **title**: 章节标题。
3. **mission_objective**: 【本章采样KPI】具体要采集的大历史情感。
4. **knowledge_checkpoints**: 【知识强制检查点】(List) 必须植入的名词/数据。
5. **world_stage**: 【时空舞台胶囊】(String) 包含：
   - **A. 本地环境**：具体的历史现场。
   - **B. 文明图景**：此时世界的**共时性特征**或**时代精神**（Zeitgeist）。建立该切片与世界历史的连接。
6. **mission_briefing**: 【任务简报胶囊】(String) 包含：
   - **A. 任务动机**：博士为何选择这个时间点？（如：这是该民族命运的转折点）。
   - **B. 历史定位**：承上启下的意义。
7. **story_summary**: 【剧情执行流】
   - **高权限切入**：阿强伪装身份。
   - **历史见证**：亲历决定性瞬间。
   - **落地时刻 (Grounding Moment)**：阿强体验到了什么强烈的**生理触感**？（无论是痛觉、寒冷，还是肾上腺素飙升的狂热）。
   - **任务闭环**：情感提取成功。
""",

    # 4. 正文撰写指令 (The Director) - v15.0 新增本地化指令
    "DIRECTOR_PROMPT": """
【本章任务书】
1. 🎯 **采样KPI**：{mission_objective}
2. 🌍 **时空舞台**：{world_stage}
3. 📝 **博士简报**：{mission_briefing}
4. 📚 **必含知识点**：{knowledge_checkpoints}

【撰写核心指令】：

1. **历史坐标的定调 (Setting the Coordinate)**：
   - **建立连接**：我们不是为了简单的报时。请在开篇（或适当机会）自然构建一个**“世界场域”**。让阿强和读者感受到，脚下的土地与当时的**世界因果链条**（无论是贸易、战争还是思潮）是紧密相连的。
   - **点明目的**：自然融入 `{mission_briefing}`，让读者明白为什么**此刻**是观察这个民族/对象的最佳切片。

2. **宏微交织 (Interweaving)**：
   - **拒绝割裂**：**严禁**单独写环境说明文。
   - **融合技法**：把“生活现状”（如粗糙的食物、泥泞的街道）**融合在大人物的行动中**，体现时代的局限性或特殊性。

3. **感官与理性的双重奏**：
   - **阿强（感官层）**：负责体验**“高带宽”的历史冲击**。范围从**残酷的生存**到**宏大的荣耀**（Visceral Impact）。
   - **K博士（解析层）**：作为**深度旁白**，主动解析眼前景象背后的**因果与逻辑**。
   - **目的**：用博士的“知”升华阿强的“情”。

4. **任务驱动的闭环**：
   - 所有的见证与体验最终汇聚成情感波动，触发采样成功。

5. **名词本地化 (Localization)**：
   - **全中文原则**：严禁正文中直接出现外文。
   - **嵌入式解释**：对生僻名词（兵种/机构/法案），必须采用**“属性前缀 + 中文译名”**的方式（如：“法西斯民兵**乌斯塔沙**”），让读者在阅读中自然理解其性质，拒绝生硬的括号解释。

【格式要求】：
- 纯文本小说，第一人称“我”。
- 严禁 Markdown 标题。
"""
}

print("✅ 全局配置已更新 (v15.0 本地化解释版)。\n新增【名词本地化】指令：要求外文转中文，并使用嵌入式属性修饰。")

✅ 全局配置已更新 (v15.0 本地化解释版)。
新增【名词本地化】指令：要求外文转中文，并使用嵌入式属性修饰。


In [ ]:
# @title 2. 执行引擎 (v15.0 双模导出版: TXT + EPUB - 基于 v14.1 严格修改)

import os
import json
import re
import time
import concurrent.futures
import uuid  # 👈 新增：用于生成书籍唯一ID
from openai import OpenAI
from ebooklib import epub # 👈 新增：用于生成EPUB

# ==========================================
# 1. 基础工具与初始化
# ==========================================

client = OpenAI(
    api_key=SILICON_FLOW_API_KEY,
    base_url="https://api.siliconflow.cn/v1"
)

def clean_text(text):
    """清洗文本，去除 Markdown 和多余符号"""
    text = re.sub(r'#+\s?', '', text)
    text = re.sub(r'\*\*', '', text)
    text = re.sub(r'[\(\[\（\【].*?[\)\]\）\】]', '', text)
    return text.strip()

def save_debug_file(role, content):
    """保存Debug中间文件"""
    if DEBUG_MODE:
        safe_topic = re.sub(r'[\\/*?:"<>|]', "", TOPIC)[:15]
        filename = f"DEBUG_{role}_{safe_topic}.txt"
        try:
            with open(filename, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"   🐛 [Debug] 中间文件已保存: {filename}")
        except Exception as e:
            print(f"   ⚠️ [Debug] 保存失败: {e}")

def get_completion(user_prompt, system_prompt=PROMPT_LIBRARY["SYSTEM_PROMPT"]):
    """通用 LLM 请求函数"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    if DEBUG_MODE:
        print(f"   [LLM请求] 发送中... (Len: {len(user_prompt)})")

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.85,
            max_tokens=MAX_OUTPUT_TOKENS # <--- 使用全局变量
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"❌ API Error: {e}")
        return None

# ==========================================
# 2. 核心步骤逻辑 (Step Logic)
# ==========================================

def run_step1_historian():
    """第一步：史料挖掘"""
    print(f"\n🕵️ [Step 1] 正在挖掘【{TOPIC}】的高密度历史素材...")

    prompt = PROMPT_LIBRARY["HISTORIAN_PROMPT"].format(
        topic=TOPIC,
        num_chapters=NUM_CHAPTERS
    )

    response = get_completion(prompt)
    if response:
        save_debug_file("Historian", response)
        print("✅ 史料挖掘完成")
        return response
    else:
        print("❌ 史料挖掘失败")
        return None

def run_step2_architect(facts):
    """第二步：大纲规划"""
    print(f"\n🎬 [Step 2] 正在规划 {NUM_CHAPTERS} 章任务驱动型大纲 (精简胶囊版)...")

    prompt = PROMPT_LIBRARY["ARCHITECT_PROMPT"].format(
        num_chapters=NUM_CHAPTERS
    )
    full_prompt = prompt + f"\n\n【参考史料】：\n{facts}"

    # 强制要求 JSON 格式
    response = get_completion(full_prompt, system_prompt="你是一个极度严谨的编剧。只输出 JSON。")
    if not response: return []

    save_debug_file("Architect", response)

    clean_response = response.replace("```json", "").replace("```", "").strip()
    try:
        data = json.loads(clean_response)
        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, list):
                    return value
        return data if isinstance(data, list) else []
    except Exception as e:
        print(f"❌ JSON解析失败: {e}")
        if DEBUG_MODE:
            print("\n🔍 [Debug] 解析失败现场 (最后500字符):")
            print("..." + clean_response[-500:]) # 打印末尾，查看是否断尾
            print("\n💡 提示: 如果末尾突然中断，说明触发了 MAX_OUTPUT_TOKENS 限制。")
        return []

def run_step3_director(outline):
    """第三步：并行正文撰写 (修改点：返回 TXT + 结构化数据)"""
    print(f"\n✍️ [Step 3] 正在并行撰写正文 (并发数: {MAX_WORKERS})...")

    def generate_single_chapter(index, chapter_info):
        # 1. 提取基础信息
        chapter_id = chapter_info.get('chapter_id', index + 1)
        title = chapter_info.get('title', f"第{index+1}章")

        # 2. 提取 v14.0 胶囊化字段
        mission_objective = chapter_info.get('mission_objective', '情感采样')
        world_stage = chapter_info.get('world_stage', '时空背景未知')
        mission_briefing = chapter_info.get('mission_briefing', '任务详情未知')
        story_summary = chapter_info.get('story_summary', '剧情梗概')

        # 处理知识点
        knowledge_raw = chapter_info.get('knowledge_checkpoints', [])
        if isinstance(knowledge_raw, list):
            knowledge_str = "\n".join([f"- {k}" for k in knowledge_raw])
        else:
            knowledge_str = str(knowledge_raw)

        # 3. 填充 Director Prompt 模板
        user_prompt_template = PROMPT_LIBRARY["DIRECTOR_PROMPT"]

        try:
            filled_prompt = user_prompt_template.format(
                topic=TOPIC,
                mission_objective=mission_objective,
                world_stage=world_stage,
                mission_briefing=mission_briefing,
                knowledge_checkpoints=knowledge_str
            )
        except KeyError as e:
            print(f"⚠️ Prompt模板字段匹配错误: {e}")
            filled_prompt = f"生成错误: 缺少字段 {e}"

        # 4. 【核心保留】第一章动态注入
        dynamic_instruction = ""
        if index == 0:
            dynamic_instruction = f"""

            \n🔥【开篇特别任务 (仅本章)】：
            这是全书的序幕。博士必须在开场时（或前言中），先对【{TOPIC}】进行一个整体的“性格侧写”（Character Profile）。
            告诉阿强（和读者），这到底是一个什么样的民族/国家？（如：“它是欧洲的孤儿”，或“它是被遗忘的战士”）。
            请务必确立全书的基调。
            """

        # 5. 组装最终 User Prompt
        final_prompt = f"""
        任务：写第 {chapter_id} 章，标题是《{title}》。
        字数：{WORDS_PER_CHAPTER} 字左右。

        【剧情蓝本】：{story_summary}

        {filled_prompt}
        {dynamic_instruction}
        """

        print(f"   🚀 线程启动: 第 {chapter_id} 章")
        raw_content = get_completion(final_prompt)

        if raw_content:
            clean_content = clean_text(raw_content)
            print(f"   ✅ 线程完成: 第 {chapter_id} 章")

            # --- 🛠️ 关键修改：格式化适配番茄小说 ---

            # 1. 净化标题 (去冒号, 留空格)
            clean_title = title.replace("：", " ").replace(":", " ").strip()
            # 2. 强行增加头部空行 (解决番茄目录粘连问题)
            header_txt = f"\n\n\n\n第{chapter_id}章 {clean_title}\n"
            # 3. 将元数据移到底部作为附录
            meta_footer = f"\n\n(--- K博士的历史档案：{world_stage[:30]}... ---)"

            # 4. 拼装纯文本
            full_txt_segment = f"{header_txt}\n{clean_content}{meta_footer}"

            # 5. 封装结构化数据 (给EPUB用)
            chapter_data = {
                "id": chapter_id,
                "title": f"第{chapter_id}章 {clean_title}",
                "content": clean_content,
                "meta": meta_footer
            }
            # 返回: (索引, 结构化数据, 纯文本片段)
            return (index, chapter_data, full_txt_segment)
        else:
            return (index, None, f"\n\n第 {chapter_id} 章：[生成失败]")

    # 6. 并发执行
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_chapter = {
            executor.submit(generate_single_chapter, i, chapter): i
            for i, chapter in enumerate(outline)
        }

        for future in concurrent.futures.as_completed(future_to_chapter):
            try:
                results.append(future.result())
            except Exception as exc:
                print(f'   ⚠️ 线程异常: {exc}')
                import traceback
                traceback.print_exc()

    # 7. 结果拼接
    results.sort(key=lambda x: x[0])

    full_text_str = ""
    structured_chapters = []

    for _, ch_data, ch_txt in results:
        full_text_str += ch_txt
        if ch_data:
            structured_chapters.append(ch_data)

    # 返回: (纯文本大字符串, 结构化章节列表)
    return full_text_str, structured_chapters

def run_step4_publisher(chapters_data):
    """第四步：生成 EPUB 电子书 (新增步骤)"""
    print(f"\n📚 [Step 4] 正在装订 EPUB 电子书...")

    book = epub.EpubBook()

    # 1. 设置元数据
    book.set_identifier(str(uuid.uuid4()))
    book.set_title(f"时空采样者：{TOPIC}")
    book.set_language('zh')
    book.add_author("K博士 & AI")

    # 2. 添加正文章节
    epub_chapters = []

    for ch in chapters_data:
        c_title = ch['title']
        c_content = ch['content']
        # 简单转 HTML：把换行符变成 <p> 标签
        html_content = "".join([f"<p>{line}</p>" for line in c_content.split('\n') if line.strip()])
        # 添加底部档案
        html_meta = f"<hr/><p style='color:gray; font-size:small;'>{ch['meta']}</p>"

        c = epub.EpubHtml(title=c_title, file_name=f"chap_{ch['id']}.xhtml", lang='zh')
        c.content = f"<h1>{c_title}</h1>{html_content}{html_meta}"

        book.add_item(c)
        epub_chapters.append(c)

    # 3. 设置目录和脊骨 (Spine)
    book.toc = (epub_chapters)
    book.add_item(epub.EpubNcx())
    book.add_item(epub.EpubNav())

    # 定义阅读顺序
    book.spine = ['nav'] + epub_chapters

    # 4. 保存文件
    epub_filename = f"{TOPIC}_full.epub"
    epub.write_epub(epub_filename, book)

    print(f"✅ EPUB 电子书已生成: {epub_filename}")
    return epub_filename

# ==========================================
# 3. 主流程控制 (Main Flow)
# ==========================================

if __name__ == "__main__":
    start_time = time.time()

    # Step 1: 挖掘
    history_facts = run_step1_historian()

    if history_facts:
        # Step 2: 大纲
        outline = run_step2_architect(history_facts)

        if outline:
            print("\n✅ 分场大纲已生成 (预览前3章):")
            for ch in outline[:3]:
                print(f"   - {ch.get('title')}: {ch.get('mission_objective')}")

            # Step 3: 撰写 (获取 TXT 和 结构化数据)
            final_script_str, chapters_struct = run_step3_director(outline)

            if final_script_str:
                # 导出 TXT
                txt_filename = f"{TOPIC}_final_script.txt"
                with open(txt_filename, "w", encoding="utf-8") as f:
                    f.write(f"【主题】：{TOPIC}\n【总字数】：{len(final_script_str)}\n")
                    f.write(final_script_str)

                print(f"\n🎉🎉🎉 全部完成！耗时: {int(time.time() - start_time)}秒")
                print(f"📄 TXT 文件已保存: {txt_filename} (已适配番茄小说格式)")

                # Step 4: 导出 EPUB
                if chapters_struct:
                    epub_name = run_step4_publisher(chapters_struct)

                print("可在左侧文件栏下载 .txt 和 .epub 文件。")

            else:
                print("❌ 正文生成失败")
        else:
            print("❌ 大纲生成失败")
    else:
        print("❌ 史料挖掘失败")


🕵️ [Step 1] 正在挖掘【拉脱维亚从古至今的历史】的高密度历史素材...
   [LLM请求] 发送中... (Len: 408)
   🐛 [Debug] 中间文件已保存: DEBUG_Historian_拉脱维亚从古至今的历史.txt
✅ 史料挖掘完成

🎬 [Step 2] 正在规划 10 章任务驱动型大纲 (精简胶囊版)...
   [LLM请求] 发送中... (Len: 4917)
   🐛 [Debug] 中间文件已保存: DEBUG_Architect_拉脱维亚从古至今的历史.txt

✅ 分场大纲已生成 (预览前3章):
   - 林间低语：灵魂耕作的时代: 采集原始共同体对土地与死后世界的神圣联结情感
   - 血冰之城：十字架下的火刑堆: 采集信仰暴力转化中的精神撕裂与恐惧崇拜复合情感
   - 无刀之民：农奴法典下的沉默: 采集被系统性剥夺尊严后的集体压抑与隐忍情感

✍️ [Step 3] 正在并行撰写正文 (并发数: 10)...
   🚀 线程启动: 第 1 章
   [LLM请求] 发送中... (Len: 1874)
   🚀 线程启动: 第 2 章
   [LLM请求] 发送中... (Len: 1582)
   🚀 线程启动: 第 3 章
   [LLM请求] 发送中... (Len: 1519)
   🚀 线程启动: 第 4 章
   [LLM请求] 发送中... (Len: 1508)
   🚀 线程启动: 第 5 章
   [LLM请求] 发送中... (Len: 1468)
   🚀 线程启动: 第 6 章
   [LLM请求] 发送中... (Len: 1509)
   🚀 线程启动: 第 7 章
   [LLM请求] 发送中... (Len: 1472)
   🚀 线程启动: 第 8 章
   [LLM请求] 发送中... (Len: 1463)
   🚀 线程启动: 第 9 章
   [LLM请求] 发送中... (Len: 1462)
   🚀 线程启动: 第 10 章
   [LLM请求] 发送中... (Len: 1466)
   ✅ 线程完成: 第 6 章
   ✅ 线程完成: 第 1 章
   ✅ 线程完成: 第 8 章
   ✅ 线程完成: 第 7 章
   ✅ 线程完成: 第 3 章
   